In [ ]:
!pip install python-pptx
!apt-get install -y libreoffice poppler-utils
!pip install pdf2image

In [ ]:
import os
import re
import yaml
import glob
import zipfile
import subprocess
from PIL import Image
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from google.colab import files

def setup_environment():
    print("🔧 Installing system dependencies (LibreOffice, Poppler, etc.)...")
    subprocess.run(['apt-get', 'update'], capture_output=True)
    subprocess.run(['apt-get', 'install', '-y', 'libreoffice', 'poppler-utils', 'fontconfig'], capture_output=True)
    subprocess.run(['pip', 'install', 'python-pptx', 'pdf2image', 'PyYAML'], capture_output=True)

def install_fonts(font_folder="/content"):
    print("📋 Checking for custom fonts to maintain presentation style...")
    font_files = []
    for ext in ['*.ttf', '*.otf', '*.TTF', '*.OTF']:
        font_files.extend(glob.glob(os.path.join(font_folder, ext)))

    if not font_files:
        print("ℹ️ No custom fonts found in /content. Using system defaults.")
        return

    custom_fonts_dir = "/usr/share/fonts/truetype/custom"
    os.makedirs(custom_fonts_dir, exist_ok=True)

    for f in font_files:
        subprocess.run(['cp', f, custom_fonts_dir], check=True)
        print(f"  + Installed: {os.path.basename(f)}")

    subprocess.run(['fc-cache', '-fv'], capture_output=True)
    print("✅ Font cache updated.")

def apply_dithering(img):
    # Adaptive palette reduction to keep colors punchy on Minecraft maps
    return img.convert('P', palette=Image.ADAPTIVE, dither=Image.FLOYDSTEINBERG).convert('RGBA')

def extract_slide_number(hyperlink, current, total):
    if not hyperlink: return None
    link_str = str(hyperlink).lower()
    nums = re.findall(r'\d+', link_str)
    if "slide" in link_str and nums: return int(nums[0])
    if any(w in link_str for w in ["next", "siguiente"]): return current + 1 if current < total else None
    if any(w in link_str for w in ["prev", "anterior", "back"]): return current - 1 if current > 1 else None
    return None

# --- Main Execution ---
setup_environment()
from pdf2image import convert_from_path

print("\n📤 Upload your PPTX (and any .ttf/.otf fonts you used):")
uploaded = files.upload()
if not uploaded: raise Exception("No files uploaded.")

pptx_file = next((f for f in uploaded.keys() if f.endswith('.pptx')), None)
if not pptx_file: raise Exception("No PPTX file found in upload.")

install_fonts()

print(f"\n--- Minecraft InteractiveBoard Setup: {pptx_file} ---")
m_wide = int(input("How many maps wide (128px each)? ") or 9)
m_high = int(input("How many maps high (128px each)? ") or 5)
bg_prefix = input("Background image name prefix (e.g., bg_): ") or "background_scene"
btn_prefix = input("Button name prefix (e.g., btn_): ") or "button"
btn_color = input("Button fill color (e.g., none or HEX): ") or "none"
btn_border = input("Button border color (e.g., none or HEX): ") or "none"
dither_on = input("Apply dithering for better map gradients? (y/n): ").lower() == 'y'
output_name = input("Resulting board/zip file name: ") or "InteractiveBoard"

P_WIDTH, P_HEIGHT = m_wide * 128, m_high * 128

# 1. High-Fidelity PNG Conversion
print("\n🖼️  Rendering slides to high-res PNG (300 DPI)...")
temp_dir = "export_temp"
if os.path.exists(temp_dir):
    import shutil
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir)

subprocess.run(['libreoffice', '--headless', '--convert-to', 'pdf', '--outdir', '.', pptx_file])
pdf_path = pptx_file.replace(".pptx", ".pdf")
slides = convert_from_path(pdf_path, dpi=300)

for i, slide in enumerate(slides, start=1):
    img = slide.resize((P_WIDTH, P_HEIGHT), Image.Resampling.LANCZOS)
    if dither_on:
        img = apply_dithering(img)
    img.save(os.path.join(temp_dir, f"{bg_prefix}{i}.png"), "PNG")
    print(f"  ✓ Processed slide {i}")

# 2. Extract Logic with Fixed YAML Nesting
print("\n📑 Mapping buttons and generating nested YAML...")
prs = Presentation(pptx_file)
pw, ph = prs.slide_width, prs.slide_height
config = {"scenes": {}} # This is the root

for i, slide in enumerate(prs.slides, start=1):
    scene_data = {
        "bg": {
            "type": "image", "position": "0 0", "size": f"{P_WIDTH} {P_HEIGHT}",
            "image": {"name": f"{bg_prefix}{i}.png", "refresh": False}
        }
    }

    b_idx = 1
    for shape in slide.shapes:
        if shape.shape_type == MSO_SHAPE_TYPE.GROUP: continue
        try:
            if shape.click_action.hyperlink and shape.click_action.hyperlink.address:
                target = extract_slide_number(shape.click_action.hyperlink.address, i, len(prs.slides))
                if target:
                    x, y = int(shape.left/pw*P_WIDTH), int(shape.top/ph*P_HEIGHT)
                    w, h = int(shape.width/pw*P_WIDTH), int(shape.height/ph*P_HEIGHT)
                    scene_data[f"{btn_prefix}{b_idx}"] = {
                        "type": "button", "position": f"{x} {y}", "size": f"{w} {h}",
                        "color": btn_color, "outline-color": btn_border,
                        "on-click": {f"switch_{b_idx}": {"type": "switch_scene", "scene": f"scene{target}"}}
                    }
                    b_idx += 1
        except: continue

    # NESTING FIX: Assign the scene data TO the scenes key
    config["scenes"][f"scene{i}"] = scene_data

# Save the final YML
with open(os.path.join(temp_dir, f"{output_name}.yml"), "w") as f:
    yaml.dump(config, f, sort_keys=False)

# 3. Zip and Download
zip_file = f"{output_name}.zip"
with zipfile.ZipFile(zip_file, 'w') as z:
    for f in os.listdir(temp_dir): z.write(os.path.join(temp_dir, f), f)

print(f"\n✅ All fixed! Indentation is correct and your {zip_file} is ready.")
files.download(zip_file)